<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/claude%2Fkind-volta-wlb7e5/llama_cpp_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# llama.cpp on Colab L4 / A100 / G4

Detects the GPU, builds llama.cpp for it (and caches the build on Google Drive), downloads a GGUF model that fits the card, and serves it with an OpenAI-compatible API, a web UI, and the DeepSeek Harness (dsh) agent front end.

Runtime -> Change runtime type -> **L4**, **A100** or **G4 GPU**. For a T4, use `llama_cpp_t4.ipynb` instead (it is tuned for partial CPU offload).

## 1. Detect the GPU

In [ ]:
import os, re, subprocess
os.environ["PATH"] += ":/usr/local/cuda/bin"

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
if smi.returncode or not smi.stdout.strip():
    raise RuntimeError("No NVIDIA GPU found. Runtime -> Change runtime type -> choose an L4, A100 or G4 GPU.")
GPU_NAME, mem_mib, cc = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(",")]
VRAM_GB   = int(float(mem_mib)) / 1024
CUDA_ARCH = cc.replace(".", "")                      # 8.0 -> 80, 8.9 -> 89, 12.0 -> 120
TIER = "xl" if VRAM_GB >= 80 else "l" if VRAM_GB >= 36 else "m" if VRAM_GB >= 20 else "s"
print(f"{GPU_NAME}: {VRAM_GB:.0f} GB VRAM, compute capability {cc} (sm_{CUDA_ARCH}), size tier '{TIER}'")
!free -h | head -2
!df -h /content | tail -1

## 2. Config (the model is picked from the GPU's memory; override any value)

In [ ]:
REPO_ID = "unsloth/Qwen3.8-27B-GGUF"

# File sizes (GB): UD-Q4_K_M 16.5 | UD-Q5_K_M 19.8 | UD-Q6_K 22.0 | Q8_0 29.1 | UD-Q8_K_XL 31.5 | BF16 (2 shards) 54.7
PRESETS = {
    "xl": dict(file="Qwen3.8-27B-Q8_0.gguf",         ctx=65536),   # 80+ GB: G4 (96 GB), A100 80 GB
    "l":  dict(file="Qwen3.8-27B-Q8_0.gguf",         ctx=32768),   # 36-79 GB: A100 40 GB
    "m":  dict(file="Qwen3.8-27B-UD-Q4_K_M.gguf",    ctx=16384),   # 20-35 GB: L4 (24 GB). UD-Q5_K_M (19.8 GB) is tighter but better
    "s":  dict(file="Qwen3.8-27B-UD-Q4_K_M.gguf",    ctx=8192),    # under 20 GB (T4): only part of the model fits; prefer llama_cpp_t4.ipynb
}

# Override to choose your own. For a split model use a pattern, e.g. "BF16/Qwen3.8-27B-BF16-*.gguf" (any GGUF repo works).
MODEL_FILE     = None
CTX_SIZE       = None
N_GPU_LAYERS   = None    # None = all layers if the model fits, otherwise an estimate from model size and VRAM
N_LAYERS_TOTAL = 64      # layer count of the 27B model, used only for the partial-offload estimate
REBUILD_LLAMA  = False   # True = ignore the cached build, pull the latest llama.cpp, rebuild, and re-save it

MODEL_FILE = MODEL_FILE or PRESETS[TIER]["file"]
CTX_SIZE   = CTX_SIZE or PRESETS[TIER]["ctx"]
print("Model:", REPO_ID, MODEL_FILE, "| context:", CTX_SIZE)

## 3. Get llama.cpp: restore the build from Google Drive (first run builds with CUDA for this GPU, then saves it)

In [ ]:
import os, re, subprocess
from google.colab import drive
drive.mount("/content/drive")

BUILD_DIR = "/content/drive/MyDrive/llm_models/builds"
os.makedirs(BUILD_DIR, exist_ok=True)

# Blackwell (sm_120, the G4) needs CUDA 12.8+; with an older toolkit fall back to PTX built for sm_90, which the driver JIT-compiles.
nvcc = subprocess.run("nvcc --version", shell=True, capture_output=True, text=True).stdout
m = re.search(r"release (\d+)\.(\d+)", nvcc)
if not m:
    raise RuntimeError("nvcc (the CUDA compiler) was not found on this runtime.")
CUDA_VER = (int(m.group(1)), int(m.group(2)))
ARCH_CMAKE = CUDA_ARCH
if int(CUDA_ARCH) >= 120 and CUDA_VER < (12, 8):
    ARCH_CMAKE = "90-virtual"
    print(f"CUDA {CUDA_VER[0]}.{CUDA_VER[1]} cannot compile sm_{CUDA_ARCH}; using {ARCH_CMAKE} (works, may be a little slower).")

TARBALL = f"{BUILD_DIR}/llama.cpp_cuda_sm{ARCH_CMAKE}.tar"
SRC, BIN = "/content/llama.cpp", "/content/llama.cpp/build/bin"

def works():
    try:
        return subprocess.run([f"{BIN}/llama-server", "--version"], capture_output=True, timeout=120).returncode == 0
    except Exception:
        return False

if not REBUILD_LLAMA and os.path.exists(TARBALL):
    print("Restoring cached build from Drive...")
    os.makedirs(f"{SRC}/build", exist_ok=True)
    subprocess.run(["tar", "-xf", TARBALL, "-C", f"{SRC}/build"], check=True)

if works():
    print("Cached build works.")
else:
    print(f"No usable cached build; building for sm_{ARCH_CMAKE} (several minutes)...")
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        open("/content/llama_build.log", "a").write(r.stdout + r.stderr)
        if r.returncode:
            raise RuntimeError(f"Failed: {cmd}\n" + (r.stdout + r.stderr)[-3000:])
    open("/content/llama_build.log", "w").close()
    sh(f"rm -rf {SRC} && git clone --depth 1 https://github.com/ggml-org/llama.cpp {SRC}")
    sh(f"cd {SRC} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={ARCH_CMAKE} -DLLAMA_CURL=OFF")
    sh(f"cd {SRC} && cmake --build build --config Release -j $(nproc) --target llama-server llama-cli")
    rev = subprocess.run(f"cd {SRC} && git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
    open(f"{SRC}/build/bin/BUILD_INFO.txt", "w").write(f"llama.cpp {rev}, sm_{ARCH_CMAKE}, CUDA {CUDA_VER[0]}.{CUDA_VER[1]}\n")
    subprocess.run(["tar", "-cf", TARBALL + ".part", "-C", f"{SRC}/build", "bin"], check=True)
    os.rename(TARBALL + ".part", TARBALL)
    print("Saved build to Drive:", TARBALL)

assert os.path.exists(f"{BIN}/llama-server"), "Build failed; see /content/llama_build.log"
print(open(f"{BIN}/BUILD_INFO.txt").read() if os.path.exists(f"{BIN}/BUILD_INFO.txt") else "")

## 4. Get the model (from Google Drive if saved, otherwise download once and save)

In [ ]:
import glob, shutil

DRIVE_DIR = "/content/drive/MyDrive/llm_models"   # permanent copy
LOCAL_DIR = "/content/models"                     # fast local disk; llama.cpp loads from here
os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(LOCAL_DIR, exist_ok=True)

on_drive = sorted(glob.glob(f"{DRIVE_DIR}/{MODEL_FILE}"))
local    = sorted(glob.glob(f"{LOCAL_DIR}/{MODEL_FILE}"))

def copy_atomic(a, b):
    os.makedirs(os.path.dirname(b), exist_ok=True)
    shutil.copyfile(a, b + ".part")      # .part so a cut-off copy is never mistaken for a full file
    os.rename(b + ".part", b)

if not on_drive:
    %pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    print("Not on Drive yet; downloading from Hugging Face...")
    snapshot_download(repo_id=REPO_ID, allow_patterns=[MODEL_FILE], local_dir=LOCAL_DIR)
    local = sorted(glob.glob(f"{LOCAL_DIR}/{MODEL_FILE}"))
    assert local, f"{MODEL_FILE} not found in {REPO_ID}"
    for f in local:
        copy_atomic(f, f"{DRIVE_DIR}/{os.path.relpath(f, LOCAL_DIR)}")
    print("Saved to Drive:", DRIVE_DIR)
elif len(local) < len(on_drive):
    print("Copying from Drive to local disk (faster to load than reading from Drive)...")
    for f in on_drive:
        dst = f"{LOCAL_DIR}/{os.path.relpath(f, DRIVE_DIR)}"
        if not os.path.exists(dst):
            copy_atomic(f, dst)
    local = sorted(glob.glob(f"{LOCAL_DIR}/{MODEL_FILE}"))

MODEL_PATH = local[0]                                  # for split files llama.cpp loads the rest from the first shard
MODEL_GB   = sum(os.path.getsize(f) for f in local) / 1e9
print(MODEL_PATH, f"({MODEL_GB:.1f} GB, {len(local)} file(s))")
!df -h /content | tail -1

## 5. Start the server in the background (also serves a built-in chat web UI)

In [ ]:
import subprocess, time, requests, secrets

API_KEY = secrets.token_urlsafe(24)   # required by every request, including through the public tunnel

if N_GPU_LAYERS is None:
    if MODEL_GB + 3 <= VRAM_GB * 0.95:                       # weights plus about 3 GB for cache and buffers
        N_GPU_LAYERS = 99
    else:                                                   # partial offload; an estimate, raise it until you run out of memory
        N_GPU_LAYERS = max(0, int(N_LAYERS_TOTAL * (VRAM_GB * 0.9 - 3) / MODEL_GB))
print("GPU layers:", N_GPU_LAYERS, "(99 = all)")

server = subprocess.Popen(
    [f"{BIN}/llama-server", "-m", MODEL_PATH, "-ngl", str(N_GPU_LAYERS), "-c", str(CTX_SIZE),
     "--host", "127.0.0.1", "--port", "8080", "--api-key", API_KEY, "--jinja"],   # --jinja: tool calls come back in OpenAI format
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(300):                       # large models take a while to load
    if server.poll() is not None:
        print("server exited:\n", open("/content/server.log").read()[-4000:]); break
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])

## 6. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8080/v1/chat/completions",
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"messages": [{"role": "user", "content": prompt}],
                            "max_tokens": max_tokens, "temperature": temperature}, timeout=900)
    r.raise_for_status()
    j = r.json(); m = j["choices"][0]["message"]
    speed = j.get("timings", {}).get("predicted_per_second")
    if speed: print(f"[{speed:.1f} tokens/s]")
    return (("[thinking]\n" + m["reasoning_content"] + "\n[answer]\n") if m.get("reasoning_content") else "") + (m.get("content") or "")

print(chat("Explain what a GGUF file is in two sentences."))

## 7. Open the web UI and the API from outside Colab

llama-server has its own chat web UI at `/`. This opens a free Cloudflare tunnel so you can use it from your browser, phone, or any OpenAI-compatible client.
The URL is public but every request needs the API key. The UI asks for it under Settings. Anyone with both the URL and key can use your GPU, so don't share them.

In [ ]:
import subprocess, re, time, os, urllib.request

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)

def open_tunnel(port):
    """Expose a local port through a free Cloudflare quick tunnel and return (process, public URL)."""
    log_path = f"/content/cloudflared_{port}.log"
    proc = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                            stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError("No tunnel URL yet:\n" + open(log_path).read()[-2000:])

In [ ]:
tunnel, PUBLIC_URL = open_tunnel(8080)
print("Web UI :", PUBLIC_URL)
print("API    :", PUBLIC_URL + "/v1")
print("API key:", API_KEY)
print(f"""
# From any machine:
from openai import OpenAI
client = OpenAI(base_url="{PUBLIC_URL}/v1", api_key="{API_KEY}")
print(client.chat.completions.create(model="local", messages=[{{"role": "user", "content": "Hello"}}]).choices[0].message.content)

# Or with curl:
curl {PUBLIC_URL}/v1/chat/completions -H "Authorization: Bearer {API_KEY}" -H "Content-Type: application/json" \\
  -d '{{"messages":[{{"role":"user","content":"Hello"}}]}}'
""")

## 8. DeepSeek Harness (dsh) front end

[dsh](https://github.com/deepseek-ai/deepseek-harness) is DeepSeek's open-source agent harness: a coding agent with a local web UI that works with any OpenAI-compatible server. This points it at the server started above and opens it through a tunnel.

- **Security:** dsh's agent can run shell commands and edit files on this Colab VM, and **Google Drive is mounted** here. Keep dsh's tool-approval prompts on, and don't approve anything touching `/content/drive` unless you mean to. The tunnel URL is public; dsh protects itself with a random access token that is part of the link printed below, so share that link with nobody.
- dsh needs Node 22.19+, which this installs (Colab's default is older).
- Agent tool calling only works if the server has the right flags (`--jinja` for llama.cpp, `--enable-auto-tool-choice --tool-call-parser ...` for vLLM). The server cell above sets them.
- An agent needs a long context (at least 16384 tokens); the context size is set from the GPU in the config cell.

In [ ]:
import os, re, subprocess, urllib.request, time, socket, textwrap

# ---- what to connect dsh to ----
BACKEND  = "http://127.0.0.1:8080"            # the OpenAI-compatible server started above
PROVIDER = "llamacpp"
MODEL    = "local"
CTX      = CTX_SIZE                  # must match the server's context size
MAXTOK   = min(4096, CTX // 2)

# ---- Node 22 (dsh needs 22.19+ or 24+) ----
NODE_DIR = "/content/node"
if not os.path.exists(f"{NODE_DIR}/bin/node"):
    base = "https://nodejs.org/dist/latest-v22.x/"
    fn = re.search(r"node-v22[\d.]+-linux-x64\.tar\.xz", urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()).group(0)
    urllib.request.urlretrieve(base + fn, "/content/node.tar.xz")
    os.makedirs(NODE_DIR, exist_ok=True)
    subprocess.run(["tar", "-xJf", "/content/node.tar.xz", "-C", NODE_DIR, "--strip-components=1"], check=True)
NPM_CACHE = "/content/drive/MyDrive/llm_models/builds/npm_cache"   # on Drive so dsh is not re-downloaded every session
os.makedirs(NPM_CACHE, exist_ok=True)
ENV = dict(os.environ, PATH=f"{NODE_DIR}/bin:" + os.environ["PATH"], npm_config_cache=NPM_CACHE, LOCAL_LLM_API_KEY=API_KEY)
print("node", subprocess.run(["node", "--version"], env=ENV, capture_output=True, text=True).stdout.strip())

# ---- dsh provider config ----
os.makedirs(os.path.expanduser("~/.dsh"), exist_ok=True)
open(os.path.expanduser("~/.dsh/settings.yaml"), "w").write(f"""llm-pi-ai:
  providers:
    {PROVIDER}:
      displayName: {PROVIDER}
      apiKeyEnv: LOCAL_LLM_API_KEY
      api: openai-completions
      baseURL: {BACKEND}/v1
      compat:
        supportsDeveloperRole: false
        maxTokensField: max_tokens
      models:
        - id: {MODEL}
          name: {MODEL}
          contextWindow: {CTX}
          maxTokens: {MAXTOK}
agent-default-model:
  provider: {PROVIDER}
  model: {MODEL}
""")

# ---- start dsh (web UI listens on 127.0.0.1:3080) ----
os.makedirs("/content/workspace", exist_ok=True)
dsh = subprocess.Popen(["npx", "-y", "@deepseek-ai/dsh", "web", "--no-open"], cwd="/content/workspace", env=ENV,
                       stdout=open("/content/dsh.log", "w"), stderr=subprocess.STDOUT)
def wait_for_port(port, proc, seconds=180):
    for _ in range(seconds):
        if proc.poll() is not None:
            return False
        try:
            socket.create_connection(("127.0.0.1", port), timeout=1).close()
            return True
        except OSError:
            time.sleep(1)
    return False

if not wait_for_port(3080, dsh):
    print(open("/content/dsh.log").read()[-3000:])
    raise RuntimeError("dsh did not start; see the log above")

# ---- publish through the tunnel; dsh protects itself with a random token in the link ----
token = re.search(r"token=([\w-]+)", open("/content/dsh.log").read()).group(1)
dsh_tunnel, DSH_URL = open_tunnel(3080)
print("Open this link (it contains the access token; treat it like a password):")
print(f"{DSH_URL}/?token={token}")
print("If the model is not preselected, pick provider", PROVIDER, "and model", MODEL, "in the UI. Logs: /content/dsh.log")

## 9. Cleanup and VRAM check

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
# server.terminate()   # stop the server when finished

## Notes
- Qwen3.8-27B is a hybrid thinking model, so replies may start with a reasoning block and use many tokens.
- The model is picked from the GPU's memory (section 2). On an L4 the 4-bit file fits entirely on the GPU; on an A100 or G4 an 8-bit file does. Set `MODEL_FILE` to try another quantization, or point `REPO_ID` at any other GGUF repo.
- Models and the compiled llama.cpp are cached on Drive (`MyDrive/llm_models`, `MyDrive/llm_models/builds`). The build is stored per GPU architecture, so switching GPU types builds once more. Set `REBUILD_LLAMA = True` to update llama.cpp.
- On a G4 (Blackwell), a CUDA toolkit older than 12.8 makes the build fall back to a JIT-compiled architecture. The cell prints a message when it does.
- The tunnel and API key last only for this session. Run the tunnel cell again for a new URL after a restart.
- Build errors are in `/content/llama_build.log`; server errors in `/content/server.log`.